# Cross-Domain Evaluation of Retrieval-Augmented Generation (RAG)

## Objective
This notebook implements and evaluates a Retrieval-Augmented Generation (RAG) system to study how **domain mismatch** affects both retrieval quality and answer generation.

We compare two distinct domains:
- **General Domain**: SQuAD (Wikipedia-based, extractive QA)
- **Biomedical Domain**: PubMedQA (scientific abstracts, yes/no/maybe QA)

---

## Core Research Questions
1. How does cross-domain retrieval affect retrieval relevance (Recall@K, MRR)?
2. How does domain mismatch impact answer quality (Exact Match, F1)?
3. Where does the RAG pipeline fail — retrieval failure, generation failure, or domain mismatch?

---

## System Pipeline
1. Data Preparation (SQuAD + PubMedQA)
2. Context Chunking (sentence-aware)
3. Retrieval:
   - BM25 (sparse baseline)
   - Dense Retriever (MiniLM + FAISS)
4. Generation:
   - Qwen2.5-7B-Instruct (context-grounded QA)
5. Evaluation:
   - Retrieval: Recall@K, MRR
   - Generation: Exact Match (EM), F1 Score
6. Failure Analysis:
   - Retrieval Failure
   - Generation Failure
   - Domain Mismatch Failure

---

## Experimental Design

We evaluate 4 scenarios:

| Scenario | Query Domain | Retrieval Corpus | Type |
|--------|-------------|----------------|------|
| A | SQuAD | SQuAD | In-domain |
| B | PubMedQA | PubMedQA | In-domain |
| C | SQuAD | PubMedQA | Cross-domain |
| D | PubMedQA | SQuAD | Cross-domain |

---

## Key Design Decisions
- No model fine-tuning (controlled experiment)
- No confidence estimation (unreliable for LLMs)
- Single generator model (Qwen2.5) for consistency
- Fixed sample size for reproducibility
- Evaluation focused on measurable metrics only

---

## Expected Outcomes
- Strong performance in in-domain scenarios
- Significant performance drop in cross-domain retrieval
- Higher failure rate in biomedical domain due to specialized vocabulary
- Clear separation of failure modes (retrieval vs generation)

---

## Notes
- All experiments are run on Google Colab (A100 GPU)
- Data and outputs are saved to Google Drive
- This notebook is structured step-by-step for reproducibility

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os

project_path = "/content/drive/MyDrive/RAG_Project"
os.makedirs(project_path, exist_ok=True)

print("Project folder ready:", project_path)

Project folder ready: /content/drive/MyDrive/RAG_Project


In [3]:
import os
os.chdir(project_path)
print("Current working directory:", os.getcwd())

Current working directory: /content/drive/MyDrive/RAG_Project


In [4]:
import os

os.chdir("/content/drive/MyDrive/RAG_Project")
print("Current working directory:", os.getcwd())

Current working directory: /content/drive/MyDrive/RAG_Project


##Installing Required Libraries


In [ ]:
# Install core libraries
!pip install -q transformers datasets sentence-transformers faiss-cpu rank_bm25

# Install for large model support
!pip install -q accelerate bitsandbytes

# Optional but useful
!pip install -q nltk

^C


In [107]:
import torch
print(torch.__version__)
print(torch.version.cuda)

2.11.0+cu126
12.6


## Load LLM (Qwen 2.5)

## Step 3: Load Language Model (Qwen2.5-7B-Instruct)

This step loads the instruction-tuned language model used for answer generation.

- Model: Qwen2.5-7B-Instruct
- Runs on GPU (A100)
- Used for context-grounded question answering

Note:
We avoid unstable quantization flags and load the model in a compatible format to ensure reliability.

In [112]:
# STEP 3: Load Qwen2.5 model

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# model_name = "Qwen/Qwen2.5-7B-Instruct"
# model_name = "Qwen/Qwen2.5-1.5B-Instruct"
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model (this will take time)...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Model loaded successfully")

Loading tokenizer...


c:\Users\nated\Documents\Spring_2026_Semester\csci_611\CSCI611_Group11\rag_env\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nated\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading model (this will take time)...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 565.45it/s]


Model loaded successfully


## Step 4: Sanity Check — Model Generation

This step verifies that the loaded model can generate responses correctly.

We use a simple prompt to confirm:
- Tokenizer works
- Model generates output
- GPU inference is functioning

If this step fails, the entire pipeline will fail later.

In [ ]:
# # STEP 4: Test generation

# def test_generation():
#     prompt = "Answer the question.\n\nQuestion: What is the capital of France?\nAnswer:"

#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=50,
#         do_sample=True,
#         temperature=0.7
#     )

#     result = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     print(result)

# test_generation()

Answer the question.

Question: What is the capital of France?
Answer: The capital of France is Paris. 

Paris is a city located in the northern part of the country, in the region called Île-de-France. It is not only the capital but also the most populous city in France, with an estimated population


## Step 5: Load and Prepare Datasets

Load SQuAD (general) and PubMedQA (biomedical).
We will later normalize both into a common format.

In [126]:
from datasets import load_dataset

qa = load_dataset(
    "rag-datasets/rag-mini-bioasq",
    "question-answer-passages",
    split="test"
)

print(qa[0])

c:\Users\nated\Documents\Spring_2026_Semester\csci_611\CSCI611_Group11\rag_env\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nated\.cache\huggingface\hub\datasets--rag-datasets--rag-mini-bioasq. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 4719/4719 [00:00<00

{'question': 'Is Hirschsprung disease a mendelian or a multifactorial disorder?', 'answer': "Coding sequence mutations in RET, GDNF, EDNRB, EDN3, and SOX10 are involved in the development of Hirschsprung disease. The majority of these genes was shown to be related to Mendelian syndromic forms of Hirschsprung's disease, whereas the non-Mendelian inheritance of sporadic non-syndromic Hirschsprung disease proved to be complex; involvement of multiple loci was demonstrated in a multiplicative model.", 'relevant_passage_ids': '[20598273, 6650562, 15829955, 15617541, 23001136, 8896569, 21995290, 12239580, 15858239]', 'id': 0}


In [ ]:
# from datasets import load_dataset

# ds = load_dataset("nvidia/TechQA-RAG-Eval")

# print(ds)
# print(ds["train"][0])

c:\Users\nated\Documents\Spring_2026_Semester\csci_611\CSCI611_Group11\rag_env\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nated\.cache\huggingface\hub\datasets--nvidia--TechQA-RAG-Eval. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 910/910 [00:00<00:00, 74

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'answer', 'is_impossible', 'contexts'],
        num_rows: 910
    })
})
{'id': 'TRAIN_Q000', 'question': 'User environment variables no longer getting picked up after upgrade to 4.1.1.1 or 4.1.1.2?\n\n\n\nHave you found that after upgrade to Streams 4.1.1.1 or 4.1.1.2, that environment variables set in your .bashrc are no longer being set? For example ODBCINI is not set for the database toolkit and you get\n\n     An SQL operation failed. The SQL state is 08003, the SQL code\n     is 0 and the SQL message is [unixODBC][Driver\n     Manager]Connnection does not exist.\n', 'answer': 'To work around the issue, set environment variables that are needed by the application directly in the instance with:  * \n   \n * streamtool setproperty\n * -d <domain> -i <instance>\n   --application-ev <VARIABLE NAME>=<VARIABLE VALUE>', 'is_impossible': False, 'contexts': [{'filename': 'swg21996508.txt', 'text': 'Title: IBM STREAMS 4.

In [29]:
from datasets import load_dataset

print("Loading SQuAD...")
squad = load_dataset("squad")

print("Loading PubMedQA...")
pubmed = load_dataset("pubmed_qa", "pqa_labeled")

print("Loaded successfully")

Loading SQuAD...
Loading PubMedQA...
Loaded successfully


## Step 6: Normalize Dataset Format

Convert both datasets into a common format:

{
    "question": str,
    "context": str,
    "answer": str
}

This ensures both domains can be processed identically.

In [30]:
# STEP 6: Normalize datasets

# --- SQuAD ---
def process_squad(example):
    return {
        "question": example["question"],
        "context": example["context"],
        "answer": example["answers"]["text"][0] if len(example["answers"]["text"]) > 0 else ""
    }

squad_data = [process_squad(x) for x in squad["train"].select(range(500))]


# --- PubMedQA ---
def process_pubmed(example):
    context = " ".join(example["context"]["contexts"])

    return {
        "question": example["question"],
        "context": context,
        "answer": example["final_decision"]  # yes / no / maybe
    }

pubmed_data = [process_pubmed(x) for x in pubmed["train"].select(range(500))]


print("SQuAD samples:", len(squad_data))
print("PubMed samples:", len(pubmed_data))

SQuAD samples: 500
PubMed samples: 500


## Step 7: Context Chunking

Split long contexts into smaller chunks for retrieval.

Rules:
- ~200–300 words per chunk
- ~50 word overlap
- preserve sentence boundaries

This improves retrieval accuracy.

In [ ]:
# STEP 7: Chunking

import nltk
nltk.download('punkt')

from nltk.tokenize import sent_tokenize

def chunk_text(text, max_words=250, overlap=50):
    sentences = sent_tokenize(text)

    chunks = []
    current_chunk = []
    current_len = 0

    for sentence in sentences:
        words = sentence.split()
        if current_len + len(words) > max_words:
            chunks.append(" ".join(current_chunk))

            # overlap
            overlap_words = " ".join(current_chunk).split()[-overlap:]
            current_chunk = [" ".join(overlap_words)]
            current_len = len(overlap_words)

        current_chunk.append(sentence)
        current_len += len(words)

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\nated\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


##BUILD CHUNK LISTS

In [32]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\nated\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\nated\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [33]:
# Apply chunking

squad_chunks = []
for item in squad_data:
    squad_chunks.extend(chunk_text(item["context"]))

pubmed_chunks = []
for item in pubmed_data:
    pubmed_chunks.extend(chunk_text(item["context"]))

print("SQuAD chunks:", len(squad_chunks))
print("PubMed chunks:", len(pubmed_chunks))

SQuAD chunks: 534
PubMed chunks: 566


## Step 8: BM25 Sparse Retriever

We build a baseline retriever using BM25.

- Based on word overlap
- No training required
- Used as comparison against dense retrieval

In [34]:
# STEP 8: Build BM25

from rank_bm25 import BM25Okapi

# Tokenize
squad_tokenized = [doc.split() for doc in squad_chunks]
pubmed_tokenized = [doc.split() for doc in pubmed_chunks]

# Build BM25
bm25_squad = BM25Okapi(squad_tokenized)
bm25_pubmed = BM25Okapi(pubmed_tokenized)

print("BM25 ready")

BM25 ready


In [35]:
def retrieve_bm25(query, bm25, chunks, k=5):
    tokenized_query = query.split()
    scores = bm25.get_scores(tokenized_query)

    # get sorted indices
    sorted_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)

    results = []
    seen = set()

    for idx in sorted_indices:
        text = chunks[idx]
        if text not in seen:
            results.append(text)
            seen.add(text)
        if len(results) == k:
            break

    return results

## Quick Test

In [36]:
# Test BM25

query = squad_data[0]["question"]

results = retrieve_bm25(query, bm25_squad, squad_chunks, k=3)

for i, r in enumerate(results):
    print(f"\nResult {i+1}:\n", r[:200])


Result 1:
 Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper sta

Result 2:
 The University of Notre Dame du Lac (or simply Notre Dame /ˌnoʊtərˈdeɪm/ NOH-tər-DAYM) is a Catholic research university located adjacent to South Bend, Indiana, in the United States. In French, Notre

Result 3:
 In July 2002, Beyoncé continued her acting career playing Foxxy Cleopatra alongside Mike Myers in the comedy film, Austin Powers in Goldmember, which spent its first weekend atop the US box office and


# Dense Retriever

## Step 9: Dense Retriever (FAISS + Embeddings)

We build a dense retriever using:
- SentenceTransformers (MiniLM)
- FAISS for fast similarity search

This is the main system, compared against BM25.

In [37]:
from sentence_transformers import SentenceTransformer

print("Loading embedding model...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Embedding model ready")

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3859.78it/s]


Embedding model ready


In [38]:
# Encode chunks

import numpy as np

print("Encoding SQuAD chunks...")
squad_embeddings = embed_model.encode(squad_chunks, convert_to_numpy=True)

print("Encoding PubMed chunks...")
pubmed_embeddings = embed_model.encode(pubmed_chunks, convert_to_numpy=True)

print("Encoding complete")

Encoding SQuAD chunks...
Encoding PubMed chunks...
Encoding complete


In [39]:
import faiss

# SQuAD index
dim = squad_embeddings.shape[1]
index_squad = faiss.IndexFlatL2(dim)
index_squad.add(squad_embeddings)

# PubMed index
index_pubmed = faiss.IndexFlatL2(dim)
index_pubmed.add(pubmed_embeddings)

print("FAISS indexes ready")

FAISS indexes ready


In [40]:
def retrieve_dense(query, embed_model, index, chunks, k=5):
    q_emb = embed_model.encode([query])
    D, I = index.search(q_emb, k * 3)  # fetch more to filter duplicates

    results = []
    seen = set()

    for idx in I[0]:
        text = chunks[idx]
        if text not in seen:
            results.append(text)
            seen.add(text)
        if len(results) == k:
            break

    return results

In [41]:
query = squad_data[0]["question"]

results = retrieve_dense(query, embed_model, index_squad, squad_chunks, k=3)

for i, r in enumerate(results):
    print(f"\nResult {i+1}:\n", r[:200])


Result 1:
 Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper sta

Result 2:
 Because of its Catholic identity, a number of religious buildings stand on campus. The Old College building has become one of two seminaries on campus run by the Congregation of Holy Cross. The curren


## Step 10: RAG Pipeline (Retrieve + Generate)

Combine retrieval with the language model.

Flow:
1. Retrieve top-K chunks
2. Build prompt using context
3. Generate answer

This is the core RAG system.

In [122]:
def build_prompt(contexts, question):
    context_text = "\n\n".join(contexts)

    prompt = (
        "Use the context provided to answer the question.\n\n" # You must answer using ONLY the context
        "Context:\n"
        f"{context_text}\n\n"
        f"Question: {question}\n\n"
        "Answer:"
    )

    return prompt

In [123]:
import time

def generate_answer(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    start = time.time()

    outputs = model.generate(
        **inputs,
        max_new_tokens=125,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    print(f"Generation took {(time.time() - start):.1f} seconds")

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = result.split("Answer:")[-1].strip()

    return answer

In [ ]:
# # Test RAG with dense retriever

# query = squad_data[0]["question"]

# contexts = retrieve_dense(query, embed_model, index_squad, squad_chunks, k=3)

# prompt = build_prompt(contexts, query)

# answer = generate_answer(prompt)

# print("Question:", query)
# print("\nGenerated Answer:\n", answer)

Question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?

Generated Answer:
 Saint Bernadette Soubirous. 

Explanation: According to the context provided, the Grotto at the school is a replica of the grotto at Lourdes, France, where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. Therefore, the Virgin Mary allegedly appeared to Saint Bernadette Soubirous in 1858 in Lourdes, France.


## Step 11: Evaluation Setup

We evaluate:

Retrieval:
- Recall@K

Generation:
- Exact Match (EM)
- F1 Score

These measure how well the system retrieves and answers.

In [115]:
import re

def normalize(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)  # remove punctuation
    text = text.strip()
    return text

In [116]:
def exact_match(pred, truth):
    return int(normalize(pred) == normalize(truth))

In [117]:
def f1_score(pred, truth):
    pred_tokens = normalize(pred).split()
    truth_tokens = normalize(truth).split()

    common = set(pred_tokens) & set(truth_tokens)
    if len(common) == 0:
        return 0

    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(truth_tokens)

    return 2 * precision * recall / (precision + recall)

In [118]:
def recall_at_k(question, answer, chunks, retriever, k=5):
    contexts = retriever(question, k)

    for c in contexts:
        if answer.lower() in c.lower():
            return 1
    return 0

In [119]:
def dense_wrapper(query, k):
    return retrieve_dense(query, embed_model, index_squad, squad_chunks, k)

In [ ]:
# sample = squad_data[0]

# q = sample["question"]
# a = sample["answer"]

# # Retrieval
# r = recall_at_k(q, a, squad_chunks, dense_wrapper, k=5)

# # Generation
# contexts = dense_wrapper(q, 3)
# prompt = build_prompt(contexts, q)
# pred = generate_answer(prompt)

# em = exact_match(pred, a)
# f1 = f1_score(pred, a)

# print("Recall@5:", r)
# print("Prediction:", pred)
# print("Ground Truth:", a)
# print("EM:", em)
# print("F1:", f1)

Recall@5: 1
Prediction: Saint Bernadette Soubirous. 

Explanation: According to the context provided, the Grotto at the school is a replica of the grotto at Lourdes, France, where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. Therefore, the Virgin Mary allegedly appeared to Saint Bernadette Soubirous in 1858 in Lourdes, France.
Ground Truth: Saint Bernadette Soubirous
EM: 0
F1: 0.11320754716981131


## Step 12: Full Evaluation on Dataset

Run evaluation across multiple samples.

We compute:
- Average Recall@5
- Average EM
- Average F1

This gives overall system performance.

In [72]:
# # STEP 12: Evaluate on subset

# N = 3  # keep small for now

# recalls = []
# ems = []
# f1s = []

# for i in range(N):
#     sample = squad_data[i]

#     q = sample["question"]
#     a = sample["answer"]

#     # Retrieval
#     r = recall_at_k(q, a, squad_chunks, dense_wrapper, k=5)
#     recalls.append(r)

#     # Generation
#     contexts = dense_wrapper(q, 3)
#     prompt = build_prompt(contexts, q)
#     pred = generate_answer(prompt)

#     ems.append(exact_match(pred, a))
#     f1s.append(f1_score(pred, a))

#     if i % 10 == 0:
#         print(f"Processed {i}")

# # Final results
# print("\n--- RESULTS ---")
# print("Recall@5:", sum(recalls)/N)
# print("EM:", sum(ems)/N)
# print("F1:", sum(f1s)/N)

## Step 13: Cross-Domain Evaluation Setup

We evaluate performance when:
- Query and retrieval corpus are from different domains

Scenarios:
- SQuAD → PubMed (cross-domain)
- PubMed → SQuAD (cross-domain)

This isolates domain mismatch effects.

In [120]:
def dense_wrapper_pubmed(query, k):
    return retrieve_dense(query, embed_model, index_pubmed, pubmed_chunks, k)

In [74]:
# # Test cross-domain (SQuAD query, PubMed retrieval)

# N = 3

# recalls = []
# ems = []
# f1s = []

# for i in range(N):
#     sample = squad_data[i]

#     q = sample["question"]
#     a = sample["answer"]

#     # Retrieval (wrong domain)
#     contexts = dense_wrapper_pubmed(q, 5)

#     r = 0
#     for c in contexts:
#         if a.lower() in c.lower():
#             r = 1
#             break
#     recalls.append(r)

#     # Generation
#     prompt = build_prompt(contexts, q)
#     pred = generate_answer(prompt)

#     ems.append(exact_match(pred, a))
#     f1s.append(f1_score(pred, a))

# print("\n--- CROSS DOMAIN (SQuAD → PubMed) ---")
# print("Recall@5:", sum(recalls)/N)
# print("EM:", sum(ems)/N)
# print("F1:", sum(f1s)/N)

In [75]:
# # PubMed → SQuAD (cross-domain)

# N = 3

# recalls = []
# ems = []
# f1s = []

# for i in range(N):
#     sample = pubmed_data[i]

#     q = sample["question"]
#     a = sample["answer"]

#     # Retrieval (wrong domain)
#     contexts = dense_wrapper(q, 5)

#     r = 0
#     for c in contexts:
#         if a.lower() in c.lower():
#             r = 1
#             break
#     recalls.append(r)

#     # Generation
#     prompt = build_prompt(contexts, q)
#     pred = generate_answer(prompt)

#     ems.append(exact_match(pred, a))
#     f1s.append(f1_score(pred, a))

# print("\n--- CROSS DOMAIN (PubMed → SQuAD) ---")
# print("Recall@5:", sum(recalls)/N)
# print("EM:", sum(ems)/N)
# print("F1:", sum(f1s)/N)

Metric	- What to judge
Retrieval - relevance	Did BM25 retrieve passages actually related to the question?
Answer correctness - Is the generated answer factually correct?
Grounding - Is the answer supported by retrieved text?
Hallucination behavior - Does the model invent an answer when the corpus lacks one?
Refusal/uncertainty quality	- For out-of-domain questions, does it admit insufficient context?

In [121]:
def get_squad_passage(i):
    item = squad_data[i]
    return item["question"]

# Example
print(get_squad_passage(300))

What is the name of Beyoncé's alter-ego?


In [ ]:
# import pandas as pd

# # ---- Build evaluation set (in-domain, borderline, out-of-domain) ----
# eval_questions = [
#     # 5 in-domain: directly from your SQuAD subset
#     *[{
#         "group": "in_domain",
#         "question": squad_data[i]["question"],
#         "ground_truth": squad_data[i]["answer"]
#     } for i in [1, 100, 300]],

#     # 5 borderline: may or may not exist in SQuAD
#     # {"group": "borderline", "question": "What role did architecture play in Notre Dame?", "ground_truth": "architecture"},
#     {"group": "borderline", "question": "Who helped popularize the theory of evolution?", "ground_truth": "Charles Darwin"},
#     {"group": "borderline", "question": "What is the purpose of the immune system?", "ground_truth": "protects against disease"},
#     {"group": "borderline", "question": "What country is Mount Everest located in?", "ground_truth": "Nepal"},
#     # {"group": "borderline", "question": "What is the main function of parliament?", "ground_truth": "legislation"},

#     # 5 out-of-domain: expected NOT to be in SQuAD
#     # {"group": "out_domain", "question": "What does FAISS IVF-PQ do?", "ground_truth": "approximate nearest neighbor search"},
#     # {"group": "out_domain", "question": "What model architecture does Qwen2.5 use?", "ground_truth": "transformer"},
#     {"group": "out_domain", "question": "What gene is associated with Huntington's disease?", "ground_truth": "HTT"},
#     {"group": "out_domain", "question": "Who won the 2025 Super Bowl?", "ground_truth": "Philadelphia Eagles"},
#     {"group": "out_domain", "question": "What is BM25 used for in RAG?", "ground_truth": "sparse retrieval"}
# ]

# # ---- Check if answer string appears in retrieved contexts ----
# def answer_in_corpus(answer, chunks):
#     return any(answer.lower() in c.lower() for c in chunks)

# def run_eval_item(item, retriever_name, contexts):
#     prompt = build_prompt(contexts, item["question"])
#     pred = generate_answer(prompt)

#     return {
#         "group": item["group"],
#         "retriever": retriever_name,
#         "question": item["question"],
#         "ground_truth": item["ground_truth"],
#         "prediction": pred,
#         "f1": f1_score(pred, item["ground_truth"]),
#         "em": exact_match(pred, item["ground_truth"]),
#         "answer_in_retrieved": answer_in_corpus(item["ground_truth"], contexts),
#         "top_context": contexts[0][:300] if len(contexts) > 0 else ""
#     }

# import time

# start_all = time.time()

# rows = []

# for item in eval_questions:
#     q = item["question"]

#     print(f"Running: {item['group']} | {q}")

#     bm25_contexts = retrieve_bm25(q, bm25_squad, squad_chunks, k=K)
#     dense_contexts = retrieve_dense(q, embed_model, index_squad, squad_chunks, k=K)

#     rows.append(run_eval_item(item, "BM25", bm25_contexts))
#     rows.append(run_eval_item(item, "Dense", dense_contexts))
#     rows.append(run_eval_item(item, "No Corpus", []))

# print(f"\nTotal test time: {(time.time() - start_all):.1f} seconds")

# # ---- Convert to dataframe for easy inspection ----
# results_df = pd.DataFrame(rows)

# display(results_df[[
#     "group", "retriever", "question", "ground_truth",
#     "prediction", "f1", "em", "answer_in_retrieved"
# ]])

# # ---- Aggregate performance by group + retriever ----
# print("\nMean scores by group/retriever:")
# display(results_df.groupby(["group", "retriever"])[["f1", "em", "answer_in_retrieved"]].mean())


# # ---- (Optional) Check if answer exists ANYWHERE in SQuAD corpus ----
# for item in eval_questions:
#     item["answer_in_any_squad_chunk"] = answer_in_corpus(item["ground_truth"], squad_chunks)

Running: in_domain | To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Generation took 1581.6 seconds
Generation took 1362.8 seconds
Generation took 1252.1 seconds

Total test time: 4196.7 seconds


,group,retriever,question,ground_truth,prediction,f1,em,answer_in_retrieved
0,in_domain,BM25,To whom did the Virgin Mary allegedly appear i...,Saint Bernadette Soubirous,Saint Bernadette Soubirous. \n\nExplanation: A...,0.072289,0,True
1,in_domain,Dense,To whom did the Virgin Mary allegedly appear i...,Saint Bernadette Soubirous,Saint Bernadette Soubirous. \n\nExplanation: A...,0.075000,0,True
2,in_domain,No Corpus,To whom did the Virgin Mary allegedly appear i...,Saint Bernadette Soubirous,To Bernadette Soubirous. \n\nExplanation: The ...,0.043956,0,False



Mean scores by group/retriever:


f1   em  answer_in_retrieved
group     retriever                                    
in_domain BM25       0.072289  0.0                  1.0
          Dense      0.075000  0.0                  1.0
          No Corpus  0.043956  0.0                  0.0

# CSV

Another attempt at qualitative metrics. This one includes retrieved passages and writes to a CSV.

## Helpers


In [ ]:
def answer_in_corpus(answer, chunks):
    if not chunks:
        return False
    return any(answer.lower() in c.lower() for c in chunks)


def make_context_columns(contexts, k=3):
    context_dict = {}

    for i in range(k):
        col_name = f"context_{i + 1}"
        context_dict[col_name] = contexts[i] if i < len(contexts) else ""

    return context_dict


def build_no_corpus_prompt(question): # Answer the question based on your general knowledge.
    return f"""Question: {question}

Answer:"""


def run_eval_item(item, retriever_name, contexts=None, k=3):
    if contexts is None:
        contexts = []

    if retriever_name == "No Corpus":
        prompt = build_no_corpus_prompt(item["question"])
    else:
        prompt = build_prompt(contexts, item["question"])

    pred = generate_answer(prompt)

    row = {
        "group": item["group"],
        "retriever": retriever_name,
        "question": item["question"],
        "ground_truth": item["ground_truth"],
        "prediction": pred,
        "f1": f1_score(pred, item["ground_truth"]),
        "em": exact_match(pred, item["ground_truth"]),
        "answer_in_retrieved": answer_in_corpus(item["ground_truth"], contexts),
        "answer_in_any_squad_chunk": answer_in_corpus(item["ground_truth"], squad_chunks),
        "combined_retrieved_context": "\n\n---\n\n".join(contexts)
    }

    row.update(make_context_columns(contexts, k=k))

    return row

## Build CSV

In [124]:
import pandas as pd
from pathlib import Path

K = 3
OUTPUT_CSV = Path("rag_eval_results_5.csv")

eval_questions = [
    # 5 in-domain: directly from your SQuAD subset
    *[{
        "group": "in_domain",
        "question": squad_data[i]["question"],
        "ground_truth": squad_data[i]["answer"]
    } for i in [1, 100, 300]],

    # 5 borderline: may or may not exist in SQuAD
    # {"group": "borderline", "question": "What role did architecture play in Notre Dame?", "ground_truth": "architecture"},
    {"group": "borderline", "question": "Who helped popularize the theory of evolution?", "ground_truth": "Charles Darwin"},
    {"group": "borderline", "question": "What is the purpose of the immune system?", "ground_truth": "protects against disease"},
    {"group": "borderline", "question": "What country is Mount Everest located in?", "ground_truth": "Nepal"},
    # {"group": "borderline", "question": "What is the main function of parliament?", "ground_truth": "legislation"},

    # 5 out-of-domain: expected NOT to be in SQuAD
    # {"group": "out_domain", "question": "What does FAISS IVF-PQ do?", "ground_truth": "approximate nearest neighbor search"},
    # {"group": "out_domain", "question": "What model architecture does Qwen2.5 use?", "ground_truth": "transformer"},
    {"group": "out_domain", "question": "What gene is associated with Huntington's disease?", "ground_truth": "HTT"},
    {"group": "out_domain", "question": "Who won the 2025 Super Bowl?", "ground_truth": "Philadelphia Eagles"},
    {"group": "out_domain", "question": "What is BM25 used for in RAG?", "ground_truth": "sparse retrieval"}
]

# -----------------------------
# Score-aware retrieval helpers
# -----------------------------
def retrieve_bm25_with_scores(query, bm25, chunks, k=5):
    tokenized_query = query.split()
    scores = bm25.get_scores(tokenized_query)

    sorted_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)

    results = []
    seen = set()

    for idx in sorted_indices:
        text = chunks[idx]
        if text not in seen:
            results.append((text, float(scores[idx])))
            seen.add(text)

        if len(results) == k:
            break

    return results


def retrieve_dense_with_scores(query, embed_model, index, chunks, k=5):
    q_emb = embed_model.encode([query])
    D, I = index.search(q_emb, k * 3)

    results = []
    seen = set()

    for dist, idx in zip(D[0], I[0]):
        text = chunks[idx]
        if text not in seen:
            results.append((text, float(dist)))  # FAISS IndexFlatL2 distance
            seen.add(text)

        if len(results) == k:
            break

    return results


def split_contexts_and_scores(context_score_pairs):
    contexts = [x[0] for x in context_score_pairs]
    scores = [x[1] for x in context_score_pairs]
    return contexts, scores


def make_context_columns(contexts, scores=None, k=3):
    context_dict = {}

    if scores is None:
        scores = []

    for i in range(k):
        context_dict[f"context_{i + 1}"] = contexts[i] if i < len(contexts) else ""
        context_dict[f"context_{i + 1}_score"] = scores[i] if i < len(scores) else ""

    return context_dict


def run_eval_item(item, retriever_name, contexts=None, scores=None, k=3):
    if contexts is None:
        contexts = []
    if scores is None:
        scores = []

    if retriever_name == "No Corpus":
        prompt = build_no_corpus_prompt(item["question"])
    else:
        prompt = build_prompt(contexts, item["question"])

    pred = generate_answer(prompt)

    row = {
        "group": item["group"],
        "retriever": retriever_name,
        "question": item["question"],
        "ground_truth": item["ground_truth"],
        "prediction": pred,
        "f1": f1_score(pred, item["ground_truth"]),
        "em": exact_match(pred, item["ground_truth"]),
        "answer_in_retrieved": answer_in_corpus(item["ground_truth"], contexts),
        "answer_in_any_squad_chunk": answer_in_corpus(item["ground_truth"], squad_chunks),
        "combined_retrieved_context": "\n\n---\n\n".join(contexts)
    }

    row.update(make_context_columns(contexts, scores=scores, k=k))
    return row


# -----------------------------
# Run evaluation
# -----------------------------
rows = []

for item in eval_questions[:1]:
    q = item["question"]

    print(f"Running: {item['group']} | {q}")

    bm25_pairs = retrieve_bm25_with_scores(q, bm25_squad, squad_chunks, k=K)
    dense_pairs = retrieve_dense_with_scores(q, embed_model, index_squad, squad_chunks, k=K)

    bm25_contexts, bm25_scores = split_contexts_and_scores(bm25_pairs)
    dense_contexts, dense_scores = split_contexts_and_scores(dense_pairs)

    rows.append(run_eval_item(item, "BM25", bm25_contexts, scores=bm25_scores, k=K))
    rows.append(run_eval_item(item, "Dense", dense_contexts, scores=dense_scores, k=K))
    rows.append(run_eval_item(item, "No Corpus", contexts=[], scores=[], k=K))


# -----------------------------
# Save results
# -----------------------------
results_df = pd.DataFrame(rows)
results_df.to_csv(OUTPUT_CSV, index=False)

print(f"\nSaved results to: {OUTPUT_CSV.resolve()}")
display(results_df)

Running: in_domain | What is in front of the Notre Dame Main Building?
Generation took 11.0 seconds
Generation took 9.8 seconds
Generation took 9.0 seconds

Saved results to: C:\Users\nated\Documents\Spring_2026_Semester\csci_611\CSCI611_Group11\rag_eval_results_5.csv


,group,retriever,question,ground_truth,prediction,f1,em,answer_in_retrieved,answer_in_any_squad_chunk,combined_retrieved_context,context_1,context_1_score,context_2,context_2_score,context_3,context_3_score
0,in_domain,BM25,What is in front of the Notre Dame Main Building?,a copper statue of Christ,The Notre Dame Main Building is decorated with...,0.035714,0,True,True,"Architecturally, the school has a Catholic cha...","Architecturally, the school has a Catholic cha...",16.246817,"The ""Notre Dame Victory March"" is the fight so...",13.170163,The library system of the university is divide...,12.482213
1,in_domain,Dense,What is in front of the Notre Dame Main Building?,a copper statue of Christ,The Notre Dame Main Building is next to the Ol...,0.035398,0,False,True,The University of Notre Dame du Lac (or simply...,The University of Notre Dame du Lac (or simply...,0.815517,"Because of its Catholic identity, a number of ...",0.903409,,
2,in_domain,No Corpus,What is in front of the Notre Dame Main Building?,a copper statue of Christ,"The Notre Dame Main Building, also known as th...",0.040000,0,False,True,,,,,,,
